# HEALPix Parent-Child Cell Visualization Notebook


This notebook demonstrates HEALPix hierarchical relationships between different resolution levels using Sentinel-2 satellite data.

In [33]:
import os
import glob
import natsort
import xarray as xr
import sys
import numpy as np

# HEALPix Parent-Child Cell Visualization

## Overview
This notebook demonstrates HEALPix hierarchical relationships between different resolution levels using Sentinel-2 satellite data.

### Key Concepts
- **Level 18**: Coarser resolution (parent cells)
- **Level 19**: Finer resolution (child cells, 4x more cells)  
- **Hierarchy**: Each Level 18 cell contains ~4 Level 19 cells

---

## 1. Imports and Setup

In [34]:
sys.path.append('../')
from data.healpix import get_chunk_info, get_bands, get_chunk, healpix_projection
from utils.plot import plot_all_chunks
from utils.healpix_plot import exploire_layer

In [17]:
# =============================================================================
# Load Sentinel-2 satellite data from Zarr format
# =============================================================================

# Dataset configuration
version = "V4"
TRAIN_DIR = f"/mnt/disk/dataset/sentinel-ai-processor/{version}/train/input"

# Find all .zarr files and load a specific one (index 11)
zarr_path = natsort.natsorted(glob.glob(os.path.join(TRAIN_DIR, "*.zarr"), recursive=False))
print(f"Found {len(zarr_path)} zarr files")
print(f"Loading: {os.path.basename(zarr_path[11])}")

# Open the Sentinel-2 datatree
dt = xr.open_datatree(zarr_path[11], engine="zarr", mask_and_scale=False, chunks={})

Found 50 zarr files
Loading: S2A_MSIL1C_20210419T103021_N0500_R108_T31TGM_20230522T074356.zarr


## 2. Load Sentinel-2 Data

In [21]:
# =============================================================================
# Extract a specific chunk from Level 19 data (finest resolution)
# =============================================================================

# Level 19 configuration - finest resolution HEALPix
level = 19  # HEALPix resolution level (NSIDE = 2^19 = 524,288)
res = "10m"  # Sentinel-2 spatial resolution
utm_crs = dt.other_metadata['horizontal_CRS_code']
chunk_size = 4096

# Chunk selection (spatial subset)
chunk_y_idx = 5  # Y-axis chunk index
chunk_x_idx = 5  # X-axis chunk index

# Get available spectral bands and chunk information
band_list = get_bands(data_tree=dt, res=res)
chunk_size_y, chunk_size_x, nb_chunks_y, nb_chunks_x = get_chunk_info(data_tree=dt, band=band_list[0], res=res)
# Extract the specific chunk
chunk= get_chunk(
    data_tree=dt, 
    res=res,
    chunk_size_y=chunk_size_y,
    chunk_size_x=chunk_size_x,
    chunk_y_idx=chunk_y_idx,
    chunk_x_idx=chunk_x_idx
).load()


chunk_level_19 = healpix_projection(chunk, level=19, utm_crs=utm_crs)
chunk_level_18 = healpix_projection(chunk, level=18, utm_crs=utm_crs)

Chunk size: y=1830, x=1830
Number of chunks: y=6, x=6
Coordinate grid shape: (1830, 1830)
X range: 791465 to 809755
Y range: 5090225 to 5108515
HEALPix Level 19 → 2,158,232 unique cells
Processing 4 variables: ['b02', 'b03', 'b04', 'b08']
Processing b02 (1/4)
Processing b03 (2/4)
Processing b04 (3/4)
Processing b08 (4/4)
HEALPix conversion complete - Level 19, 2,158,232 cells
Coordinate grid shape: (1830, 1830)
X range: 791465 to 809755
Y range: 5090225 to 5108515
HEALPix Level 18 → 540,592 unique cells
Processing 4 variables: ['b02', 'b03', 'b04', 'b08']
Processing b02 (1/4)
Processing b03 (2/4)
Processing b04 (3/4)
Processing b08 (4/4)
HEALPix conversion complete - Level 18, 540,592 cells


## 3. Configure Level 19 Processing

In [22]:
# =============================================================================
# Apply xdggs decoding for visualization
# =============================================================================

import xdggs

chunk_level_19 = chunk_level_19.pipe(xdggs.decode)
chunk_level_18 = chunk_level_18.pipe(xdggs.decode)

## 4. Convert to HEALPix Format
Process the data chunk and convert it to HEALPix grids at both Level 18 and Level 19.

In [29]:
# =============================================================================
# HEALPix Parent-Child Utility Functions
# =============================================================================

import healpy as hp

def get_parent_cell_ids(child_cell_ids, child_level, parent_level, nest=True):
    """
    Get parent HEALPix cell IDs from child cell IDs at different resolution levels.
    
    Parameters
    ----------
    child_cell_ids : array-like
        Array of HEALPix cell IDs at the child (higher) resolution level.
    child_level : int
        Resolution level of the child cells (higher number = finer resolution).
    parent_level : int  
        Target parent resolution level (lower number = coarser resolution).
    nest : bool, optional
        Whether to use nested HEALPix indexing scheme. Default: True.
        
    Returns
    -------
    numpy.ndarray
        Array of parent cell IDs corresponding to each input child cell.
        
    Examples
    --------
    >>> # Get level 18 parents from level 19 cells
    >>> child_ids = np.array([185450551767, 185450551772, 185450551776])
    >>> parent_ids = get_parent_cell_ids(child_ids, child_level=19, parent_level=18)
    >>> print(parent_ids)
    
    Notes
    -----
    - parent_level must be less than child_level (coarser resolution)
    - Uses bit-shifting for nested indexing: parent_id = child_id >> (2 * level_diff)
    - For ring indexing, converts between nested and ring schemes as needed
    - Maintains same indexing scheme (nested/ring) for output as specified
    """
    child_cell_ids = np.asarray(child_cell_ids)
    
    # Validate inputs
    if parent_level >= child_level:
        raise ValueError(f"Parent level ({parent_level}) must be less than child level ({child_level})")
    
    if parent_level < 0 or child_level < 0:
        raise ValueError("Resolution levels must be non-negative")
    
    level_diff = child_level - parent_level
    child_nside = 2 ** child_level
    parent_nside = 2 ** parent_level
    
    if nest:
        # For nested indexing, use bit-shifting (fast method)
        parent_cell_ids = child_cell_ids >> (2 * level_diff)
    else:
        # For ring indexing, convert through nested scheme
        # Ring -> Nested -> Parent Nested -> Parent Ring
        child_nested = hp.ring2nest(child_nside, child_cell_ids)
        parent_nested = child_nested >> (2 * level_diff)
        parent_cell_ids = hp.nest2ring(parent_nside, parent_nested)
    
    return parent_cell_ids

def get_all_parent_levels(child_cell_ids, child_level, nest=True):
    """
    Get parent cell IDs for all resolution levels from 0 to child_level-1.
    
    Parameters
    ----------
    child_cell_ids : array-like
        Array of HEALPix cell IDs at the finest resolution level.
    child_level : int
        Resolution level of the input cells.
    nest : bool, optional
        Whether to use nested HEALPix indexing scheme. Default: True.
        
    Returns
    -------
    dict
        Dictionary mapping parent levels to arrays of parent cell IDs.
        Keys are levels 0 to child_level-1, values are numpy arrays.
        
    Examples
    --------
    >>> child_ids = np.array([185450551767, 185450551772])
    >>> all_parents = get_all_parent_levels(child_ids, child_level=19)
    >>> print(f"Level 18 parents: {all_parents[18]}")
    >>> print(f"Level 0 parent: {all_parents[0]}")  # Should be 0-11 (base HEALPix cells)
    """
    result = {}
    
    for parent_level in range(child_level):
        result[parent_level] = get_parent_cell_ids(
            child_cell_ids, child_level, parent_level, nest=nest
        )
    
    return result


## 5. HEALPix Utility Functions
Define functions to compute parent-child relationships between different HEALPix resolution levels.

In [ ]:
# =============================================================================
# Create Interactive Visualizations (First 10 cells only)
# =============================================================================

import lonboard

# Get level 19 cell IDs (from 500th to 510th cell)
level_19_cell_ids = sorted(chunk_level_19.cell_ids.values)[500:510]
print(f"Level 19 cells (first 10): {level_19_cell_ids}")

# Get corresponding parent IDs at level 18
level_18_parent_ids = get_parent_cell_ids(level_19_cell_ids, child_level=19, parent_level=18, nest=True)
unique_level_18_parents = np.unique(level_18_parent_ids)
print(f"Level 18 parent cells: {unique_level_18_parents}")
print(f"Number of unique parents: {len(unique_level_18_parents)}")

Level 19 cells (first 10): [np.int64(185450552825), np.int64(185450552826), np.int64(185450552827), np.int64(185450552828), np.int64(185450552829), np.int64(185450552830), np.int64(185450552831), np.int64(185450552832), np.int64(185450552833), np.int64(185450552834)]
Level 18 parent cells: [46362638206 46362638207 46362638208]
Number of unique parents: 3


In [35]:
# Create Level 18 dataset with only the parent cells
level_18_data = xr.Dataset({
    'Sentinel2': xr.DataArray(
        np.arange(len(unique_level_18_parents), dtype=float),  # Different colors for each parent
        dims="cells",
        coords={
            "cell_ids": xr.DataArray(
                unique_level_18_parents,
                dims="cells",
                attrs={
                    "grid_name": "healpix",
                    "indexing_scheme": "nested",
                    "resolution": 18,
                }
            )
        }
    )
})

level_18_data["cell_ids"].attrs = level_18_data.Sentinel2.cell_ids.attrs
level_18_data = level_18_data.pipe(xdggs.decode)

# Create Level 19 dataset with only the first 10 cells
level_19_data = xr.Dataset({
    'Sentinel2': xr.DataArray(
        np.arange(len(level_19_cell_ids), dtype=float),  # Different colors for each child
        dims="cells",
        coords={
            "cell_ids": xr.DataArray(
                level_19_cell_ids,
                dims="cells",
                attrs={
                    "grid_name": "healpix",
                    "indexing_scheme": "nested", 
                    "resolution": 19,
                }
            )
        }
    )
})

level_19_data["cell_ids"].attrs = level_19_data.Sentinel2.cell_ids.attrs
level_19_data = level_19_data.pipe(xdggs.decode)

# Create interactive maps
level_18_map = lonboard.Map([
    exploire_layer(
        level_18_data.Sentinel2.compute(),
        alpha=0.80,
        cmap='plasma'  # Purple/pink colors for parents
    )
])

level_19_map = lonboard.Map([
    exploire_layer(
        level_19_data.Sentinel2.compute(),
        alpha=0.80,
        cmap='viridis'  # Green/yellow colors for children
    )
])

## 6. Create Interactive Visualizations
Generate interactive maps showing parent-child relationships using a subset of 10 cells.

In [31]:
level_18_map

Map(custom_attribution='', layers=(SolidPolygonLayer(filled=True, get_fill_color=arro3.core.ChunkedArray<Fixed…

### Level 18 Map (Parent Cells)
Interactive map showing the coarser resolution parent cells in plasma colors.

In [32]:
# Display Level 19 map
level_19_map

Map(custom_attribution='', layers=(SolidPolygonLayer(filled=True, get_fill_color=arro3.core.ChunkedArray<Fixed…

### Level 19 Map (Child Cells)
Interactive map showing the finer resolution child cells in viridis colors.

## Summary

This notebook demonstrates:

### Key Results
- **Level 19**: 2,158,232 total HEALPix cells (finest resolution)
- **Level 18**: 540,592 total HEALPix cells (coarser resolution)  
- **Selected subset**: 10 Level 19 cells → 3 unique Level 18 parents
- **Hierarchy**: Each parent contains multiple children cells

### Technical Implementation
- **Coordinate transformation**: UTM → Geographic → HEALPix indices
- **Data aggregation**: Pixel values averaged within each HEALPix cell
- **Parent-child mapping**: Bit-shifting for efficient level conversion
- **Interactive visualization**: Separate maps for each resolution level

### Visualizations
- **Plasma colormap**: Parent cells (Level 18) in purple/pink tones
- **Viridis colormap**: Child cells (Level 19) in green/yellow tones
- **Interactive features**: Zoom, pan, layer controls

---

## Notes

- The `get_parent_cell_ids()` function uses bit-shifting for efficient parent ID calculation
- HEALPix nested indexing scheme ensures spatial locality preservation  
- Interactive maps use lonboard with exploire_layer for HEALPix polygon rendering
- xdggs library provides proper coordinate decoding for visualization